# Gemini Agent Session Inference Manifest Generation

This Colab notebook prepares Gemini's session-aware transcription manifests for evaluation by merging predictions from our Vertex AI Agent Sessions pipeline with ground-truth segmentations.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/gemini_agent_session/create_inference_manifest_masked_audio.ipynb)

### Key Differences from the Standard Batch Manifest Tool

Unlike `model/colabs/gemini_create_inference_manifest.ipynb` which operates on Vertex Batch Prediction outputs, this notebook is custom-tailored to support the context-retaining Agent Sessions pipeline:
1. **Agent Sessions Output Parser**: Parses predictions directly from the clean NDJSON checkpoints created by our session transcription loop (`{"audio_filepath": "...", "transcript": "..."}`), avoiding complex nested Vertex Batch JSON schemas.
2. **Direct Segment Key Matching**: Rather than matching records using floating-point timestamps (which are prone to precision/rounding errors), this notebook uses direct `(example_id, segment_id)` lookups parsed directly from the segment FLAC filenames, ensuring **100% match accuracy**.

### Core Functions

1. **Loads Ground Truth Manifest**: Pulls the baseline ASR manifest containing segment information from GCS.
2. **Collects Session Predictions**: Downloads and aggregates session checkpoint `.jsonl` outputs from the configured GCS results folder.
3. **Resolves Exact Key Matches**: Extracts `example_id` (channel) and `segment_id` from filenames and maps transcripts directly to their target segment rows.
4. **Generates Labeled Manifest**: Outputs a unified `.jsonl` file matching the evaluation framework schema.
5. **Automated Export & Preview**: Backs up the finalized manifest to GCS and displays a premium, structured tabular preview of the merged transcriptions.


In [ ]:
# @title Install dependencies
%pip install -q loguru

In [ ]:
# @title Imports
import collections
import json
from pathlib import Path
import re
from typing import Any

from google.cloud import storage
from google.colab import auth, userdata
from loguru import logger

In [ ]:
# @title Authentication and client initialization
from google.colab import auth, userdata

print("Attempting standard browser authentication...")
auth.authenticate_user()
print("Browser authentication successful!")

GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCS_BUCKET = userdata.get("GCS_BUCKET")
!gcloud config set project {GCP_PROJECT_ID} --quiet

In [ ]:
# @title Pipeline Configuration
# fmt: off
# @markdown Base path under 'segmented_audio/' (e.g., 'echo/eval_audio' or 'one_hour_pilot_audio')
INPUT_DATASET_PATH = ""  # @param {type:"string"}
# @markdown Base path under 'transcripts/' and 'inference_manifests/' (e.g., 'echo/eval' or 'one_hour_pilot')
OUTPUT_DATASET_PATH = ""  # @param {type:"string"}
MODEL_ID = "gemini-3.1-flash-lite"  # @param ["gemini-3.1-flash-lite", "gemini-3.1-flash-lite-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview", "gemini-3.5-flash"] {type:"string"}
EXPERIMENT_NAME = ""  # @param {type:"string"}
# @markdown Enable if modifications (ie, resampling/downmixing) were applied during segmentation:
AUDIO_PREPROCESSING = False  # @param {type:"boolean"}
# @markdown Enable if masking was applied during segmentation:
AUDIO_MASKING = True  # @param {type:"boolean"}
# fmt: on

MODEL_VERSION = re.sub(r"[-\.]", "_", MODEL_ID)

assert not (AUDIO_PREPROCESSING and AUDIO_MASKING), (
    "Cannot enable both AUDIO_PREPROCESSING and AUDIO_MASKING simultaneously."
)
assert INPUT_DATASET_PATH, "INPUT_DATASET_PATH must be provided and cannot be empty."
assert OUTPUT_DATASET_PATH, "OUTPUT_DATASET_PATH must be provided and cannot be empty."
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided and cannot be empty."

# Handle Preprocessing suffix for segmented audio
GCS_INPUT_DIR = f"segmented_audio/{INPUT_DATASET_PATH}"
if not (AUDIO_PREPROCESSING or AUDIO_MASKING):
    GCS_INPUT_DIR = f"{GCS_INPUT_DIR}_raw"
if AUDIO_MASKING:
    GCS_INPUT_DIR = f"{GCS_INPUT_DIR}_masked"

# Define transcript input and inference manifest output directories
GCS_INPUT_TRANSCRIPTS_DIR = (
    f"transcripts/{OUTPUT_DATASET_PATH}/{MODEL_VERSION}/{EXPERIMENT_NAME}"
)

INFERENCE_MANIFEST_OUTPUT_DIR = (
    f"inference_manifests/{OUTPUT_DATASET_PATH}/{MODEL_VERSION}/{EXPERIMENT_NAME}"
)

# Safely convert path slashes to underscores for filenames
DATASET_PREFIX = OUTPUT_DATASET_PATH.replace("/", "_")

BATCH_MANIFEST_FILENAME = "batch_manifest.jsonl"
LABELED_SEGMENTS_FILENAME = (
    f"{DATASET_PREFIX}_transcriptions_all_annotations.json"
)
INFERENCE_MANIFEST_FILENAME = f"{DATASET_PREFIX}_{EXPERIMENT_NAME}.jsonl"

In [ ]:
# @title Download batch manifest
!gcloud storage cp gs://{GCS_BUCKET}/{GCS_INPUT_DIR}/{BATCH_MANIFEST_FILENAME} .

In [ ]:
# @title Manifest Processing Functions
def load_manifest(path: str) -> list[dict[str, Any]]:
    data = []
    if not Path(path).exists():
        logger.error(f"Manifest path not found: {path}")
        return []
    with open(path, encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return data


def merge_gcs_results_to_manifest(
    batch_manifest_data: list[dict[str, Any]],
    gcs_bucket_name: str,
    output_file: str,
) -> dict[str, Any]:
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(gcs_bucket_name)

    gemini_predictions = {}

    # Try to find the results folder
    prefix = f"{GCS_INPUT_TRANSCRIPTS_DIR}/"
    logger.info(
        f"Searching for predictions with prefix: gs://{gcs_bucket_name}/{prefix}"
    )

    blobs = list(bucket.list_blobs(prefix=prefix))
    # Only include files named exactly 'predictions.jsonl'
    valid_blobs = [
        b for b in blobs if b.name.split('/')[-1] == "predictions.jsonl" and (b.size or 0) > 0
    ]

    if not valid_blobs:
        logger.error(f"No files named 'predictions.jsonl' found in {prefix}")
        return {
            "total": len(batch_manifest_data),
            "matched": 0,
            "missing": len(batch_manifest_data),
        }

    logger.info(f"Found {len(valid_blobs)} valid 'predictions.jsonl' file(s).")
    for blob in valid_blobs:
        local_preds = "temp_preds.jsonl"
        blob.download_to_filename(local_preds)

        with open(local_preds) as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    data = json.loads(line)
                    audio_uri = data.get("audio_filepath", "")
                    if not audio_uri:
                        continue

                    filename = Path(audio_uri).stem
                    if "__seg" not in filename:
                        continue

                    example_id, seg_key = filename.split("__seg")
                    raw_text = data.get("transcript", "").strip()
                    gemini_predictions[(example_id, seg_key)] = raw_text
                except Exception as e:
                    continue

    merged_records = []
    matched_count = 0
    for b_info in batch_manifest_data:
        gemini_text = ""
        example_id = b_info.get("example_id", "")
        segment_id = b_info.get("segment_id", "")
        if (example_id, segment_id) in gemini_predictions:
            gemini_text = gemini_predictions[(example_id, segment_id)]
            matched_count += 1

        merged_records.append(
            {**b_info, f"pred_text_{MODEL_VERSION}": gemini_text}
        )

    with open(output_file, "w", encoding="utf-8") as f_out:
        f_out.writelines(json.dumps(rec) + "\n" for rec in merged_records)

    return {
        "total": len(batch_manifest_data),
        "matched": matched_count,
        "missing": len(batch_manifest_data) - matched_count,
    }

In [ ]:
# @title Merge GCS Predictions & Upload Manifest
# Re-running the pipeline with the corrected parsing logic
stats = merge_gcs_results_to_manifest(
    batch_manifest_data=load_manifest(BATCH_MANIFEST_FILENAME),
    gcs_bucket_name=GCS_BUCKET,
    output_file=INFERENCE_MANIFEST_FILENAME,
)

logger.info(
    f"Processing complete: {stats['matched']} matches found out of {stats['total']} total segments."
)

if stats["matched"] > 0:
    # Upload the final manifest back to GCS
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(GCS_BUCKET)
    gcs_output_path = (
        f"{INFERENCE_MANIFEST_OUTPUT_DIR}/{INFERENCE_MANIFEST_FILENAME}"
    )
    bucket.blob(gcs_output_path).upload_from_filename(
        INFERENCE_MANIFEST_FILENAME
    )
    logger.info(
        f"Successfully uploaded merged manifest to: gs://{GCS_BUCKET}/{gcs_output_path}"
    )

    # Premium Visual Receipt Table
    import pandas as pd
    from IPython.display import display

    merged_data = load_manifest(INFERENCE_MANIFEST_FILENAME)
    if merged_data:
        df = pd.DataFrame(merged_data)
        display(
            df[["example_id", "segment_id", f"pred_text_{MODEL_VERSION}"]].head(
                10
            )
        )